# Proyecto de Ciencia de Datos — Estimación del volumen futuro de combustibles vendidos (EESS)

**Metodologías:** análisis exploratorio · SVD · clustering (K-means, DBSCAN) · embeddings y ChromaDB · perceptrón y MLP (Keras)

**Cómo ejecutarlo en Google Colab**
1. `Entorno de ejecución → Ejecutar todo` (la primera vez, la celda 1 instala `chromadb` y `sentence-transformers`).
2. Cuando la celda de carga lo pida, sube `EESS_limpio.csv` (o déjalo en el directorio de trabajo de Colab).
3. Todo usa `SEMILLA = 42` para que los resultados sean reproducibles. Hay dos interruptores en la celda de configuración (`USAR_PRECIO_ACTUAL`, `USAR_SENTENCE_TRANSFORMERS`).

**Índice**
0. Configuración · 1. Definición del caso · 2. EDA · 3. Preparación y SVD · 4. Clustering (K-means, DBSCAN, comparación) · 5. Embeddings, similitud y ChromaDB · 6. Perceptrón y MLP · 7. Teoría de los errores · 8. Conclusiones y limitaciones

## 0. Configuración

In [ ]:
# Celda 1 · Instalación (solo Colab). chromadb y sentence-transformers no vienen preinstalados.
!pip -q install chromadb sentence-transformers

In [ ]:
# Celda 2 · Importaciones y parámetros globales
import os, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, mean_absolute_error, mean_squared_error, r2_score)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
pd.options.display.float_format = '{:,.3f}'.format
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

SEMILLA = 42
np.random.seed(SEMILLA)

# True  -> el modelo también usa el precio del MISMO período (no se conoce al pronosticar a futuro).
# False -> solo usa información disponible ANTES del período a predecir (recomendado, sin fuga de información).
USAR_PRECIO_ACTUAL = False
# True -> embeddings semánticos multilingües (requiere internet en Colab). Si falla, se usa TF-IDF + SVD (LSA).
USAR_SENTENCE_TRANSFORMERS = True

## 1. Definición del caso de estudio

**Problema.** El archivo `EESS_limpio.csv` contiene ventas mensuales de combustibles reportadas por Establecimientos de Expendio al Público (estaciones de servicio y otros operadores) en Argentina. El objetivo es **estimar el volumen de combustible que se venderá en un período futuro** para una combinación estación–producto–canal, y complementar el análisis con segmentación (clustering), representaciones vectoriales y redes neuronales.

**Preguntas de investigación**
1. ¿Qué estructura tienen las ventas (producto, canal, tipo de negocio, precio, volumen)? ¿Cuántas dimensiones realmente aportan información (SVD)?
2. ¿Existen segmentos naturales de ventas? ¿K-means (grupos esféricos) o DBSCAN (densidad) los describe mejor?
3. ¿Permiten los atributos textuales (operador, bandera, localidad) encontrar establecimientos similares y recuperarlos con una base vectorial?
4. ¿Cuánto mejora una red neuronal (perceptrón, MLP) respecto de líneas base simples al predecir el volumen de un período **posterior** al de entrenamiento?

**Unidad de análisis.** Una fila = un establecimiento (`nro_inscripcion`) × producto × canal de comercialización × período (año-mes).

**Variable objetivo.** `volumen` (continua, positiva y muy asimétrica; se modela como `log1p(volumen)`).

**Lo verificado en el archivo cargado** (se imprime de nuevo en la sección 2): 61.771 filas y 22 columnas; **solo 8 períodos distintos** (enero de 2018 a 2024 y febrero de 2022), no una serie mensual continua; sin duplicados exactos; `no_novimientos` y `sin_movimientos` son constantes; `fecha_de_baja` tiene ~92 % de nulos; `precio_surtidor` vale 0 en ~15 % de las filas.

**Limitaciones que condicionan todo el diseño**
- La **unidad de `volumen` no está documentada** en el archivo. Las magnitudes difieren mucho por producto (p. ej. GNC vs. nafta), lo que sugiere unidades o escalas distintas. No se asume ninguna: se trabaja en escala logarítmica y se reportan métricas en ambas escalas.
- Con solo enero (y un febrero) **no se puede estimar estacionalidad**; la variable `mes` es casi constante y no se usa.
- La cobertura por establecimiento es desigual (muchas claves aparecen en un solo período), así que una parte de las predicciones **no tiene historial**.
- Es una muestra: no representa necesariamente todo el mercado.
- Pueden existir valores extremos por errores de carga o de unidad; se detectan en la sección 2 pero no se eliminan sin evidencia.

## 2. Preparación y análisis exploratorio (EDA)

In [ ]:
# Celda 3 · Importación y lectura
RUTA = 'EESS_limpio.csv'
if not os.path.exists(RUTA):
    try:
        from google.colab import files
        print('Sube el archivo EESS_limpio.csv ...')
        subido = files.upload()
        RUTA = list(subido.keys())[0]
    except ImportError:
        raise FileNotFoundError('No se encontró EESS_limpio.csv en el directorio de trabajo.')

df_raw = pd.read_csv(RUTA)
print('Dimensiones:', df_raw.shape)
display(df_raw.head(3))

In [ ]:
# Celda 4 · Estructura, tipos, nulos, duplicados y períodos
info = pd.DataFrame({'dtype': df_raw.dtypes.astype(str),
                     'nulos': df_raw.isna().sum(),
                     '%nulos': df_raw.isna().mean() * 100,
                     'únicos': df_raw.nunique()})
display(info)

print('Duplicados exactos:', df_raw.duplicated().sum(),
      '| duplicados ignorando id:', df_raw.drop(columns='id').duplicated().sum())
print('Columnas constantes:', [c for c in df_raw.columns if df_raw[c].nunique(dropna=False) == 1])

por_periodo = df_raw.groupby('periodo').agg(filas=('id', 'size'),
                                            establecimientos=('nro_inscripcion', 'nunique'),
                                            volumen_total=('volumen', 'sum'))
display(por_periodo)

In [ ]:
# Celda 5 · Estadística descriptiva
num_cols = ['precio_sin_impuestos', 'precio_con_impuestos', 'precio_surtidor', 'volumen']
display(df_raw[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T)

print('\nVolumen por producto:')
display(df_raw.groupby('producto')['volumen'].describe())

print('Asimetría (skew) del volumen:', round(df_raw['volumen'].skew(), 1))
print('Filas con precio_surtidor == 0:', int((df_raw.precio_surtidor == 0).sum()))
print('\n% de precio_surtidor == 0 por canal:')
display(pd.crosstab(df_raw['canal_de_comercializacion'], df_raw.precio_surtidor == 0, normalize='index') * 100)

for c in ['bandera', 'tipo_negocio', 'canal_de_comercializacion', 'producto']:
    print(f'\n{c}: {df_raw[c].nunique()} categorías')
    display(df_raw[c].value_counts().head(8).to_frame('filas'))

### Decisiones de limpieza y transformación (justificación)

| Decisión | Motivo |
|---|---|
| Eliminar `no_novimientos` y `sin_movimientos` | Son constantes: no aportan información. |
| No usar `id`, `cuit`, `direccion`, `operador` como predictores | Son identificadores o texto de alta cardinalidad; provocarían memorización. `operador`/`localidad` se usan solo como texto para embeddings (sección 5). |
| `fecha_de_baja` solo para EDA | ~92 % de nulos y formato no estándar; no se imputa. |
| `precio_surtidor == 0` → `NaN` y no usarlo como predictor | En esas filas el cero es "sin precio al público" (canales Agro, Transporte, etc.), no un precio real. Se usa `precio_con_impuestos`, que no tiene ceros. |
| Agregar filas con la misma clave (establecimiento, producto, canal, período) | Hay pocas claves repetidas que difieren en `exentos` (exento / no exento): se suma el volumen y se promedian los precios. |
| `y = log1p(volumen)` | Distribución extremadamente asimétrica; el logaritmo estabiliza la varianza y evita que unos pocos valores dominen el entrenamiento. |
| No eliminar valores atípicos | Sin documentación de unidades no se puede afirmar que sean errores; se marcan y se analiza su efecto. |

In [ ]:
# Celda 6 · Limpieza y construcción del panel (una fila por establecimiento-producto-canal-período)
df = df_raw.copy()
df['periodo'] = pd.to_datetime(df['periodo'])
df['precio_surtidor'] = df['precio_surtidor'].replace(0, np.nan)
df['exentos_i'] = df['exentos'].astype(int)

CLAVE = ['nro_inscripcion', 'producto', 'canal_de_comercializacion']
n_dup = df.duplicated(CLAVE + ['periodo']).sum()
print('Filas con clave repetida en el mismo período:', n_dup)

panel = (df.groupby(CLAVE + ['periodo'], as_index=False)
           .agg(volumen=('volumen', 'sum'),
                precio_con_impuestos=('precio_con_impuestos', 'mean'),
                precio_sin_impuestos=('precio_sin_impuestos', 'mean'),
                precio_surtidor=('precio_surtidor', 'mean'),
                exentos_i=('exentos_i', 'max'),
                operador=('operador', 'first'), bandera=('bandera', 'first'),
                tipo_negocio=('tipo_negocio', 'first'), localidad=('localidad', 'first'),
                provincia=('provincia', 'first'), con_baja=('fecha_de_baja', lambda s: int(s.notna().any()))))

panel['y'] = np.log1p(panel['volumen'])
panel['log_precio'] = np.log(panel['precio_con_impuestos'])
panel = panel.sort_values(CLAVE + ['periodo']).reset_index(drop=True)
print('Panel:', panel.shape, '| claves únicas:', panel.groupby(CLAVE).ngroups)

In [ ]:
# Celda 7 · Distribuciones
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
vmax = panel['volumen'].quantile(0.99)
ax[0, 0].hist(panel['volumen'].clip(upper=vmax), bins=60, color='#4c72b0')
ax[0, 0].set_title('Volumen (recortado al percentil 99)')
ax[0, 1].hist(panel['y'], bins=60, color='#55a868')
ax[0, 1].set_title('log1p(volumen)')
orden = panel.groupby('producto')['y'].median().sort_values().index
sns.boxplot(data=panel, y='producto', x='y', order=orden, ax=ax[1, 0], fliersize=1)
ax[1, 0].set_title('log1p(volumen) por producto')
sns.countplot(data=panel, y='canal_de_comercializacion', ax=ax[1, 1],
              order=panel['canal_de_comercializacion'].value_counts().index)
ax[1, 1].set_title('Filas por canal')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
tmp = panel.groupby('periodo').agg(filas=('y', 'size'), y_mediana=('y', 'median'))
tmp['filas'].plot(kind='bar', ax=ax[0], color='#4c72b0'); ax[0].set_title('Filas por período (cobertura desigual)')
tmp['y_mediana'].plot(marker='o', ax=ax[1]); ax[1].set_title('Mediana de log1p(volumen) por período')
for a in ax: a.set_xticklabels([d.strftime('%Y-%m') for d in tmp.index], rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Celda 8 · Valores atípicos (regla IQR dentro de cada producto, en escala log)
def marca_iqr(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return (s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)

panel['atipico_iqr'] = panel.groupby('producto')['y'].transform(marca_iqr)
tabla = panel.groupby('producto').agg(filas=('y', 'size'), atipicos=('atipico_iqr', 'sum'))
tabla['%'] = tabla['atipicos'] / tabla['filas'] * 100
display(tabla)

print('Los 10 volúmenes más grandes (revisar si corresponden a la unidad del producto):')
display(panel.nlargest(10, 'volumen')[['periodo', 'operador', 'producto', 'canal_de_comercializacion', 'volumen', 'precio_con_impuestos']])
print('Se conservan: en escala log su influencia es acotada y no hay evidencia documental de error.')

In [ ]:
# Celda 9 · Correlaciones y relaciones entre variables
panel['log_precio_sin'] = np.log(panel['precio_sin_impuestos'])
cols_corr = ['y', 'log_precio', 'log_precio_sin', 'exentos_i']
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(panel[cols_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax[0]); ax[0].set_title('Pearson')
sns.heatmap(panel[cols_corr].corr(method='spearman'), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax[1]); ax[1].set_title('Spearman')
plt.tight_layout(); plt.show()

print('Correlación y vs log_precio DENTRO de cada producto (evita la mezcla de escalas entre productos):')
display(panel.groupby('producto').apply(lambda d: d['y'].corr(d['log_precio'])).to_frame('corr').T)
print('Nota: log_precio y log_precio_sin son casi colineales; se usa solo uno.')

**Variables útiles y excluidas por metodología**

| Metodología | Variables usadas | Excluidas |
|---|---|---|
| SVD / K-means / DBSCAN | `y` (log volumen), `log_precio`, `exentos`, producto, canal, tipo de negocio | identificadores, texto, provincia/bandera (alta cardinalidad: ensuciarían la geometría) |
| Embeddings / ChromaDB | texto = operador + bandera + tipo de negocio + localidad + provincia + producto + canal | variables numéricas (van como metadatos) |
| Perceptrón / MLP | volumen y precio **rezagados**, brecha temporal, nº de observaciones previas, producto, canal, provincia, bandera, tipo de negocio, `exentos` | `volumen` y precio del mismo período (fuga), `mes`, `anio`, identificadores |

## 3. Preparación de variables y reducción de dimensionalidad (SVD)

**Matriz para el análisis no supervisado.** Variables numéricas estandarizadas (`y`, `log_precio`, `exentos`) + one-hot de `producto`, `canal_de_comercializacion` y `tipo_negocio`. Las dummies se **centran pero no se escalan**: escalarlas daría un peso desproporcionado a las categorías raras (p. ej. las 39 filas de "Nafta común").

> **Importante:** estos clusters usan `y` (el objetivo) y son solo **descriptivos**. No se usan como variable predictora en la sección 6 porque filtrarían el valor a predecir.

**Fundamento de SVD.** Para una matriz centrada $X \in \mathbb{R}^{n\times p}$: $X = U\Sigma V^\top$, con $U$ ($n\times p$) y $V$ ($p\times p$) de columnas ortonormales y $\Sigma=\mathrm{diag}(\sigma_1\ge\dots\ge\sigma_p\ge0)$.
- Las columnas de $V$ son las **direcciones principales** (cargas) y $Z = XV = U\Sigma$ son las **coordenadas** (puntuaciones).
- La varianza explicada por el componente $j$ es $\sigma_j^2/\sum_i\sigma_i^2$ (con $X$ centrada, SVD equivale a PCA).
- La mejor aproximación de rango $k$ (teorema de Eckart–Young) es $X_k=U_k\Sigma_kV_k^\top$ y su error relativo es $\|X-X_k\|_F^2/\|X\|_F^2 = 1-\text{varianza acumulada}$.

**Criterio para elegir $k$:** menor $k$ con ≥ 90 % de varianza acumulada, contrastado con el codo del gráfico de sedimentación.

In [ ]:
# Celda 10 · Matriz de análisis y SVD
panel['exentos_f'] = panel['exentos_i'].astype(float)
NUM_CL = ['y', 'log_precio', 'exentos_f']
CAT_CL = ['producto', 'canal_de_comercializacion', 'tipo_negocio']

Xn = StandardScaler().fit_transform(panel[NUM_CL])
ohe_cl = OneHotEncoder(sparse_output=False, dtype=float)
Xc = ohe_cl.fit_transform(panel[CAT_CL])
Xc = Xc - Xc.mean(axis=0)                       # centrado sin escalado
X_cl = np.hstack([Xn, Xc])
nombres_cl = NUM_CL + list(ohe_cl.get_feature_names_out(CAT_CL))
print('Matriz de análisis:', X_cl.shape)

U, S, Vt = np.linalg.svd(X_cl, full_matrices=False)
var_exp = S**2 / np.sum(S**2)
var_acum = np.cumsum(var_exp)
K_SVD = int(np.argmax(var_acum >= 0.90) + 1)
print(f'Componentes para >=90% de varianza: {K_SVD} (varianza acumulada {var_acum[K_SVD-1]:.3f})')

Z = X_cl @ Vt[:K_SVD].T                        # coordenadas en el subespacio reducido
err_rel = 1 - var_acum[K_SVD - 1]
X_k = (U[:, :K_SVD] * S[:K_SVD]) @ Vt[:K_SVD]
print(f'Error relativo de reconstrucción (Frobenius²): {np.linalg.norm(X_cl - X_k)**2 / np.linalg.norm(X_cl)**2:.4f} = 1 - var acumulada = {err_rel:.4f}')

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].bar(range(1, len(S)+1), var_exp, color='#4c72b0'); ax[0].set_title('Sedimentación (varianza por componente)'); ax[0].set_xlabel('componente')
ax[1].plot(range(1, len(S)+1), var_acum, marker='o'); ax[1].axhline(0.90, color='r', ls='--'); ax[1].axvline(K_SVD, color='g', ls='--')
ax[1].set_title('Varianza acumulada'); ax[1].set_xlabel('componente')
carga = pd.DataFrame(Vt[:4].T, index=nombres_cl, columns=[f'CP{i+1}' for i in range(4)])
sns.heatmap(carga, cmap='coolwarm', center=0, ax=ax[2]); ax[2].set_title('Cargas de los 4 primeros componentes')
plt.tight_layout(); plt.show()

idx = np.random.RandomState(SEMILLA).choice(len(Z), 15000, replace=False)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=Z[idx, 0], y=Z[idx, 1], hue=panel['producto'].values[idx], s=8, alpha=.6)
plt.xlabel('CP1'); plt.ylabel('CP2'); plt.title('Estructura en el plano CP1–CP2 (muestra de 15.000)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8); plt.show()
display(carga.abs().sort_values('CP1', ascending=False).head(6))

**Lectura de resultados (ejecución de referencia, semilla 42).** Se pasa de **31 dimensiones a 7 componentes reteniendo ≈ 91 % de la varianza** (error relativo de reconstrucción ≈ 0,09). CP1 (≈ 29 %) está dominado por `y` y `log_precio` junto con el indicador de GNC: separa los productos de volumen enorme (GNC) y los registros de precio alto. CP3 vuelve a combinar `y` y `log_precio`, y CP4 está dominado por el tipo de negocio (solo líquidos vs. duales líquidos + GNC). Las dummies aportan dimensiones casi independientes, por eso hacen falta varios componentes; se acepta perder ≤ 10 % de varianza a cambio de un espacio más compacto, donde la distancia euclídea es más estable para K-means y DBSCAN.

## 4. Aprendizaje no supervisado: clustering

### 4.1 K-means
**Por qué:** es rápido, escala bien y entrega centroides interpretables como "perfiles" de venta. **Supuestos:** grupos aproximadamente esféricos y de tamaño similar; todas las observaciones deben pertenecer a algún grupo; sensible a atípicos.

**Elección de $k$:** método del codo (inercia $=\sum\|x-\mu_{c(x)}\|^2$) + *silhouette* ($s=\frac{b-a}{\max(a,b)}\in[-1,1]$). Se reportan también Davies-Bouldin (menor es mejor) y Calinski-Harabasz (mayor es mejor). El silhouette se calcula sobre una muestra de 10.000 puntos por costo $O(n^2)$.

In [ ]:
# Celda 11 · K-means: selección de k
rng = np.random.RandomState(SEMILLA)
idx_sil = rng.choice(len(Z), size=min(10000, len(Z)), replace=False)

filas = []
for k in range(2, 11):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z)
    filas.append(dict(k=k, inercia=km_k.inertia_,
                      silhouette=silhouette_score(Z[idx_sil], km_k.labels_[idx_sil]),
                      davies_bouldin=davies_bouldin_score(Z, km_k.labels_),
                      calinski_harabasz=calinski_harabasz_score(Z, km_k.labels_)))
sel = pd.DataFrame(filas).set_index('k')
display(sel)

# codo por distancia máxima a la cuerda que une los extremos de la curva de inercia
ks = sel.index.values.astype(float); ine = sel['inercia'].values
p1, p2 = np.array([ks[0], ine[0]]), np.array([ks[-1], ine[-1]])
dx, dy = p2 - p1
dist = [abs(dx * (p1[1] - i) - (p1[0] - k) * dy) / np.hypot(dx, dy) for k, i in zip(ks, ine)]
k_codo = int(ks[int(np.argmax(dist))])
k_sil = int(sel['silhouette'].idxmax())
print(f'k sugerido por codo: {k_codo} | k con mayor silhouette: {k_sil}')

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sel['inercia'].plot(marker='o', ax=ax[0]); ax[0].axvline(k_codo, color='r', ls='--'); ax[0].set_title('Método del codo')
sel['silhouette'].plot(marker='o', ax=ax[1], color='g'); ax[1].axvline(k_sil, color='r', ls='--'); ax[1].set_title('Silhouette')
sel['davies_bouldin'].plot(marker='o', ax=ax[2], color='purple'); ax[2].set_title('Davies-Bouldin (menor = mejor)')
plt.tight_layout(); plt.show()

# Regla de decisión: se prioriza silhouette; si codo y silhouette difieren, se usa el de mayor silhouette
# pero se exige k >= 3 para que la segmentación sea útil como descripción de mercado.
K_OPT = k_sil if k_sil >= 3 else max(k_codo, 3)
print('k elegido:', K_OPT)

In [ ]:
# Celda 12 · K-means: modelo final, visualización y caracterización
km = KMeans(n_clusters=K_OPT, n_init=20, random_state=SEMILLA).fit(Z)
panel['cl_kmeans'] = km.labels_

plt.figure(figsize=(8, 6))
sns.scatterplot(x=Z[idx, 0], y=Z[idx, 1], hue=km.labels_[idx], palette='tab10', s=8, alpha=.6)
c2 = km.cluster_centers_[:, :2]
plt.scatter(c2[:, 0], c2[:, 1], marker='X', s=200, c='k')
plt.title(f'K-means (k={K_OPT}) en CP1–CP2'); plt.xlabel('CP1'); plt.ylabel('CP2'); plt.show()

def perfil_clusters(col):
    g = panel.groupby(col)
    t = g.agg(n=('y', 'size'), vol_mediano=('volumen', 'median'), vol_medio=('volumen', 'mean'),
              y_medio=('y', 'mean'), precio_mediano=('precio_con_impuestos', 'median'), pct_exentos=('exentos_i', 'mean'))
    t['% del total'] = t['n'] / t['n'].sum() * 100
    return t

display(perfil_clusters('cl_kmeans'))
print('Composición por producto (fila = cluster):')
display(pd.crosstab(panel['cl_kmeans'], panel['producto'], normalize='index') * 100)
print('Composición por canal:')
display(pd.crosstab(panel['cl_kmeans'], panel['canal_de_comercializacion'], normalize='index') * 100)

# descripción automática: producto y canal dominantes de cada cluster
for c in sorted(panel['cl_kmeans'].unique()):
    d = panel[panel['cl_kmeans'] == c]
    pr = d['producto'].value_counts(normalize=True); ca = d['canal_de_comercializacion'].value_counts(normalize=True)
    print(f'Cluster {c}: {len(d):,} filas | producto dominante: {pr.index[0]} ({pr.iloc[0]:.0%}) | '
          f'canal dominante: {ca.index[0]} ({ca.iloc[0]:.0%}) | volumen mediano: {d.volumen.median():,.1f}')

**Interpretación (ejecución de referencia, semilla 42; los números pueden variar levemente con otras versiones de las librerías).** Con k = 5 K-means separa:
- **Cluster GNC** (~8 % de las filas): casi todo GNC, volumen mediano ≈ 81.000 (frente a ≈ 45–50 de los demás) y precio bajo (≈ 49). Es la estructura dominante y refleja la diferencia de escala del volumen por producto.
- **Cluster de registros exentos** (~3 %): reúne todas las filas con `exentos = 1`.
- **Cluster de precios altos** (~8 %): precios entre ≈ 230 y 1.999 (mediana ≈ 822) frente a ≈ 100 del resto. No está documentado a qué corresponde (otra unidad de precio, otro concepto de reporte, etc.); conviene revisarlo con la fuente.
- **Dos clusters de combustibles líquidos** (~30 % y ~50 %) que se diferencian sobre todo por el **tipo de negocio**: bocas duales (líquidos + GNC) frente a bocas de solo líquidos.

El silhouette más alto aparece en k = 2 (≈ 0,66), que solo separa GNC del resto: una partición demasiado gruesa para describir el mercado. Por eso la regla del notebook exige k ≥ 3 y se toma el codo (k = 5). Es una segmentación válida y descriptiva, pero dos de los grupos (GNC y precios altos) provienen de escalas de `volumen` y de `precio` por producto/registro, no necesariamente de comportamientos comerciales distintos.

### 4.2 DBSCAN
**Idea:** un punto es *núcleo* si tiene al menos `min_samples` vecinos (él incluido) a distancia ≤ `eps`; los clusters son componentes conexas de puntos núcleo y los puntos sin vecindad suficiente son **ruido** (`-1`). No exige fijar el número de clusters y detecta formas arbitrarias y atípicos.

- **`eps`**: radio de vecindad. Muy pequeño → casi todo es ruido; muy grande → todo se fusiona en un solo cluster. Se guía con el gráfico de **k-distancias** (distancia al k-ésimo vecino ordenada): el "codo" sugiere el valor.
- **`min_samples`**: densidad mínima. Regla práctica: ≥ dimensión + 1, comúnmente $2d$ en datos de $d$ dimensiones.

Como DBSCAN usa una sola densidad global, rinde mal si los grupos tienen densidades muy distintas, y esa es una limitación relevante aquí (productos con escalas de volumen muy diferentes).

In [ ]:
# Celda 13 · DBSCAN: gráfico de k-distancias y análisis de sensibilidad
d_dim = Z.shape[1]
MS0 = max(2 * d_dim, 10)
t0 = time.time()
nn = NearestNeighbors(n_neighbors=MS0).fit(Z)
kdist = np.sort(nn.kneighbors(Z)[0][:, -1])
print(f'k-distancias calculadas en {time.time()-t0:.1f}s (min_samples base = {MS0})')

# codo de la curva de k-distancias (distancia máxima a la cuerda, ejes normalizados)
xs = np.linspace(0, 1, len(kdist)); ys = (kdist - kdist.min()) / (kdist.max() - kdist.min() + 1e-12)
EPS0 = float(kdist[int(np.argmax(np.abs(ys - xs)))])
print(f'eps sugerido por el codo: {EPS0:.3f} | percentiles de k-distancia 50/90/95/99: '
      f'{np.percentile(kdist,[50,90,95,99]).round(3)}')

plt.figure(figsize=(8, 4.5)); plt.plot(kdist); plt.axhline(EPS0, color='r', ls='--', label=f'eps≈{EPS0:.2f}')
plt.title(f'Gráfico de k-distancias (k={MS0})'); plt.xlabel('puntos ordenados'); plt.ylabel('distancia al k-ésimo vecino'); plt.legend(); plt.show()

# Sensibilidad en una muestra (DBSCAN es costoso en memoria cuando hay muchos puntos casi idénticos)
N_MUESTRA = 15000
im = np.random.RandomState(SEMILLA).choice(len(Z), N_MUESTRA, replace=False)
Zm = Z[im]
res = []
for f_eps in [0.5, 1.0, 1.5]:
    for f_ms in [0.5, 1, 2]:
        eps, ms = EPS0 * f_eps, int(MS0 * f_ms)
        lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(Zm)
        mask = lab != -1; ncl = len(set(lab[mask]))
        sil = np.nan
        if ncl >= 2:
            ii = np.random.RandomState(SEMILLA).choice(np.where(mask)[0], min(5000, mask.sum()), replace=False)
            sil = silhouette_score(Zm[ii], lab[ii])
        res.append(dict(eps=eps, min_samples=ms, clusters=ncl, ruido_pct=(~mask).mean() * 100,
                        mayor_cluster_pct=(np.bincount(lab[mask]).max() / len(lab) * 100) if ncl else 0, silhouette=sil))
sens = pd.DataFrame(res)
display(sens)

In [ ]:
# Celda 14 · DBSCAN: configuración elegida, ajuste final, ruido y caracterización
# Regla: entre configuraciones con >=2 clusters y ruido <= 15 %, la de mayor silhouette;
# si ninguna cumple, la más cercana al eps sugerido por el codo.
cand = sens[(sens.clusters >= 2) & (sens.ruido_pct <= 15)].dropna(subset=['silhouette'])
if len(cand):
    mejor = cand.sort_values('silhouette', ascending=False).iloc[0]
else:
    mejor = sens.iloc[(sens.eps - EPS0).abs().argsort().iloc[0]]
EPS, MS = float(mejor['eps']), int(mejor['min_samples'])
print(f'Configuración elegida: eps={EPS:.3f}, min_samples={MS}')

db = DBSCAN(eps=EPS, min_samples=MS).fit(Z)
panel['cl_dbscan'] = db.labels_
n_cl_db = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f'Clusters: {n_cl_db} | ruido: {(db.labels_==-1).sum():,} ({(db.labels_==-1).mean():.1%})')

plt.figure(figsize=(8, 6))
lab_i = db.labels_[idx]
plt.scatter(Z[idx][lab_i == -1, 0], Z[idx][lab_i == -1, 1], s=6, c='lightgray', label='ruido')
sns.scatterplot(x=Z[idx][lab_i != -1, 0], y=Z[idx][lab_i != -1, 1], hue=lab_i[lab_i != -1], palette='tab10', s=8)
plt.title('DBSCAN en CP1–CP2 (gris = ruido)'); plt.xlabel('CP1'); plt.ylabel('CP2'); plt.show()

display(perfil_clusters('cl_dbscan').sort_values('n', ascending=False).head(15))
print('Composición por producto de los 15 grupos más grandes y del ruido (-1):')
top = panel['cl_dbscan'].value_counts().head(15).index.union([-1])
display(pd.crosstab(panel['cl_dbscan'], panel['producto'], normalize='index').loc[top] * 100)
pureza = (pd.crosstab(panel['cl_dbscan'], panel['producto'], normalize='index').drop(index=-1, errors='ignore').max(axis=1))
print(f'Pureza de producto: el {(pureza > 0.95).mean():.0%} de los clusters DBSCAN tiene >95% de un solo producto')
print('El ruido son observaciones raras (combinaciones poco frecuentes o volúmenes extremos):')
display(panel[panel.cl_dbscan == -1].nlargest(8, 'volumen')[['periodo', 'producto', 'canal_de_comercializacion', 'volumen', 'precio_con_impuestos']])

### 4.3 Comparación de algoritmos
Para que la comparación sea justa, las métricas internas (silhouette, Davies-Bouldin, Calinski-Harabasz) se calculan **solo sobre los puntos que DBSCAN no marcó como ruido**, y se calculan para K-means sobre ese mismo subconjunto. Estas métricas favorecen clusters compactos y convexos, por lo que **penalizan a DBSCAN por construcción**; se interpretan junto con la geometría de los datos y no como veredicto único.

In [ ]:
# Celda 15 · Comparación K-means vs DBSCAN
mask = panel['cl_dbscan'].values != -1
idx_ok = np.where(mask)[0]
ii = np.random.RandomState(SEMILLA).choice(idx_ok, min(10000, len(idx_ok)), replace=False)

def metricas_cluster(Zs, lab, ii):
    ok = len(set(lab)) >= 2
    return dict(silhouette=silhouette_score(Zs[ii], lab[ii]) if ok else np.nan,
                davies_bouldin=davies_bouldin_score(Zs, lab) if ok else np.nan,
                calinski_harabasz=calinski_harabasz_score(Zs, lab) if ok else np.nan)

comp = pd.DataFrame({
    'K-means': dict(clusters=K_OPT, obs_agrupadas=len(Z), ruido=0, **metricas_cluster(Z, panel.cl_kmeans.values, idx_sil)),
    'K-means (sin puntos-ruido de DBSCAN)': dict(clusters=len(set(panel.cl_kmeans.values[mask])), obs_agrupadas=int(mask.sum()), ruido=0,
                                                 **metricas_cluster(Z[mask], panel.cl_kmeans.values[mask], np.searchsorted(idx_ok, np.sort(ii)))),
    'DBSCAN': dict(clusters=n_cl_db, obs_agrupadas=int(mask.sum()), ruido=int((~mask).sum()),
                   **metricas_cluster(Z[mask], panel.cl_dbscan.values[mask], np.searchsorted(idx_ok, np.sort(ii)))),
}).T
display(comp)
if n_cl_db >= 1:
    print('Adjusted Rand Index entre ambas particiones (solo no-ruido):',
          round(adjusted_rand_score(panel.cl_kmeans.values[mask], panel.cl_dbscan.values[mask]), 3))
print('\nSensibilidad: K-means depende de k (y de la inicialización); DBSCAN depende de eps y min_samples (ver tabla de la celda 13).')

**Ventajas y limitaciones**

| | K-means | DBSCAN |
|---|---|---|
| Fortalezas | Rápido, asigna todas las observaciones, centroides interpretables, métricas internas bien definidas | No requiere fijar *k*, encuentra formas arbitrarias, aísla atípicos como ruido |
| Debilidades | Asume grupos esféricos y de tamaño parecido; los atípicos arrastran los centroides; hay que elegir *k* | Una sola densidad global; muy sensible a `eps`; en datos mixtos con dummies, muchos puntos casi idénticos generan un cluster gigante |
| Geometría de estos datos | Las dummies de producto/canal/tipo de negocio crean grupos compactos y bien separados, y `y`/`log_precio` forman nubes continuas dentro de cada uno → K-means los captura con pocos grupos | Con una densidad global, DBSCAN rodea cada combinación densa y la separa: produce **decenas de clusters pequeños casi puros en un producto** en lugar de segmentos de mercado |

**Decisión.** Para **segmentar el mercado** es más apropiado **K-means**: los datos son mixtos (categorías + magnitudes), la estructura dominante son pocos grupos compactos asociados al producto, al tipo de negocio y al nivel de precio, y se necesita asignar todas las observaciones. En la ejecución de referencia DBSCAN (eps ≈ 0,35, `min_samples` = 7) devolvió **110 clusters** casi puros en un producto (ARI ≈ 0,22 frente a K-means), con ≈ 1,9 % de ruido; su silhouette es menor y su índice de Calinski-Harabasz mucho menor que el de K-means, aunque su Davies-Bouldin es algo mejor. Esa fragmentación es demasiado fina para interpretar. Donde **sí** aporta DBSCAN es como **detector de observaciones raras**: su ruido contiene los volúmenes extremos (GNC de más de 800.000, gasoil de hasta ≈ 680.000 en canales Agro, Transporte de cargas y Otros sectores), candidatos a revisar por posibles errores de carga o de unidad. Se recomienda usar K-means para segmentar y DBSCAN para auditar atípicos.

## 5. Representaciones vectoriales, similitud y base de datos vectorial

**¿Hay variables textuales?** Sí, pero son **texto corto y categórico** (operador, bandera, tipo de negocio, localidad, provincia, producto, canal), no texto libre. Por eso un embedding de lenguaje aporta una similitud *semántica y ortográfica* entre nombres (p. ej. "Estación de Servicio" ≈ "Est. de Servicios", marcas y localidades parecidas), pero **no captura el comportamiento de ventas**: dos estaciones con texto parecido pueden vender volúmenes muy distintos. Por eso las variables numéricas (`y_medio`, `n_obs`) se guardan como **metadatos** y no dentro del embedding.

**Construcción de la representación.**
- *Preferida:* `sentence-transformers` (`paraphrase-multilingual-MiniLM-L12-v2`), un Transformer que convierte cada texto en un vector denso de 384 dimensiones.
- *Alternativa sin descargas:* TF-IDF de n-gramas de caracteres + SVD (LSA), un embedding clásico y determinista. La celda cae en esta opción si el modelo no puede descargarse.

Los vectores se normalizan (norma 1). **Similitud coseno:** $\cos(a,b)=\frac{a\cdot b}{\|a\|\|b\|}$; **distancia coseno** $=1-\cos(a,b)$. Con vectores unitarios el coseno es el producto punto.

In [ ]:
# Celda 16 · Documentos de texto (una fila por combinación única) y embeddings
panel['texto'] = (panel['operador'].str.title() + ' | bandera ' + panel['bandera'].str.title() + ' | ' +
                  panel['tipo_negocio'] + ' | ' + panel['localidad'].str.title() + ', ' + panel['provincia'].str.title() +
                  ' | ' + panel['producto'] + ' | canal ' + panel['canal_de_comercializacion'])

docs = (panel.groupby('texto')
             .agg(n_obs=('y', 'size'), y_medio=('y', 'mean'), provincia=('provincia', 'first'),
                  producto=('producto', 'first'), bandera=('bandera', 'first'), canal=('canal_de_comercializacion', 'first'))
             .reset_index())
docs['id'] = ['doc_' + str(i) for i in range(len(docs))]
print('Documentos únicos:', len(docs))
print('Ejemplo:', docs['texto'].iloc[0])

def construir_embedder(textos):
    if USAR_SENTENCE_TRANSFORMERS:
        try:
            from sentence_transformers import SentenceTransformer
            modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
            fn = lambda t: modelo.encode(list(t), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
            return 'sentence-transformers (MiniLM multilingüe)', fn
        except Exception as e:
            print('sentence-transformers no disponible, se usa TF-IDF + SVD. Motivo:', repr(e)[:120])
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import Normalizer
    tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)
    svd = TruncatedSVD(n_components=128, random_state=SEMILLA)
    norm = Normalizer()
    svd.fit(tfidf.fit_transform(textos))
    fn = lambda t: norm.transform(svd.transform(tfidf.transform(list(t))))
    return 'TF-IDF (n-gramas de caracteres) + SVD', fn

nombre_emb, embed = construir_embedder(docs['texto'].tolist())
t0 = time.time()
E = np.asarray(embed(docs['texto'].tolist()), dtype='float32')
print(f'Embeddings: {nombre_emb} | forma {E.shape} | {time.time()-t0:.1f}s | norma media {np.linalg.norm(E, axis=1).mean():.3f}')

In [ ]:
# Celda 17 · Similitud coseno y observaciones similares
def similares(i, k=6):
    sim = E @ E[i]                                  # vectores unitarios: producto punto = coseno
    orden = np.argsort(-sim)[1:k + 1]
    out = docs.iloc[orden][['texto', 'producto', 'provincia', 'y_medio']].copy()
    out['similitud_coseno'] = sim[orden]; out['distancia_coseno'] = 1 - sim[orden]
    return out

cand = docs[(docs.bandera == 'YPF') & (docs.producto == 'GNC') & (docs.provincia == 'BUENOS AIRES')]
ref = int(cand.index[0]) if len(cand) else 0
print('Referencia:', docs.loc[ref, 'texto'])
display(similares(ref))

# verificación contra la implementación de scikit-learn
sk = cosine_similarity(E[ref:ref + 1], E)[0]
assert np.allclose(sk, E @ E[ref], atol=1e-4), 'el producto punto no coincide con cosine_similarity'
print('OK: producto punto == cosine_similarity para vectores normalizados')

# Clustering sobre los embeddings (K-means esférico por vectores normalizados)
ks_emb, sil_emb = [4, 6, 8, 10, 12], []
for k in ks_emb:
    l = KMeans(n_clusters=k, n_init=5, random_state=SEMILLA).fit_predict(E)
    sil_emb.append(silhouette_score(E, l, metric='cosine', sample_size=min(8000, len(E)), random_state=SEMILLA))
print(dict(zip(ks_emb, np.round(sil_emb, 3))))
k_emb = ks_emb[int(np.argmax(sil_emb))]
docs['cl_emb'] = KMeans(n_clusters=k_emb, n_init=10, random_state=SEMILLA).fit_predict(E)
print('k elegido para embeddings:', k_emb)
display(pd.crosstab(docs['cl_emb'], docs['producto'], normalize='index').round(2) * 100)
display(docs.groupby('cl_emb').agg(docs=('texto', 'size'), prov_principal=('provincia', lambda s: s.value_counts().index[0]),
                                   bandera_principal=('bandera', lambda s: s.value_counts().index[0]), y_medio=('y_medio', 'mean')))

### Base de datos vectorial con ChromaDB
- **Almacenamiento:** cada documento se guarda con su *embedding*, el texto original y **metadatos** (provincia, producto, bandera, canal, `n_obs`, `y_medio`).
- **Indexación:** Chroma construye un índice **HNSW** (*Hierarchical Navigable Small World*): un grafo por capas que permite buscar vecinos aproximados en tiempo ≈ logarítmico, en lugar de comparar contra todos los vectores (búsqueda exacta, $O(n)$).
- **Recuperación por similitud:** `query` devuelve los *n* vectores más cercanos según la métrica de la colección (aquí coseno; Chroma devuelve la **distancia** $=1-\cos$).
- **Consultas vectoriales:** por texto (se embebe la consulta con el mismo modelo), por vector o por documento existente, combinables con **filtros de metadatos** (`where`).
- **Aplicaciones en este caso:** buscar establecimientos comparables, imputar o estimar el nivel de ventas de una estación nueva con los vecinos más parecidos, detectar duplicados o variantes de nombre de un mismo operador, y recuperar contexto para un asistente generativo (RAG).

In [ ]:
# Celda 18 · ChromaDB: crear colección, indexar y consultar
import chromadb

cliente = chromadb.Client()                         # base en memoria (se pierde al reiniciar el entorno)
try:
    cliente.delete_collection('eess')
except Exception:
    pass
ESPACIO = 'cosine'
try:                                                # versiones recientes de Chroma
    col = cliente.create_collection('eess', configuration={'hnsw': {'space': 'cosine'}})
except Exception:
    try:                                            # versiones anteriores
        col = cliente.create_collection('eess', metadata={'hnsw:space': 'cosine'})
    except Exception:                               # último recurso: L2 (con vectores unitarios el orden es el mismo)
        col = cliente.create_collection('eess'); ESPACIO = 'l2'
        print('Aviso: se usa distancia L2; con vectores unitarios, distancia coseno = L2² / 2')

LOTE = 2000
for i in range(0, len(docs), LOTE):
    b = docs.iloc[i:i + LOTE]
    col.add(ids=b['id'].tolist(),
            embeddings=E[i:i + LOTE].tolist(),
            documents=b['texto'].tolist(),
            metadatas=[{'provincia': r.provincia, 'producto': r.producto, 'bandera': r.bandera, 'canal': r.canal,
                        'n_obs': int(r.n_obs), 'y_medio': float(r.y_medio)} for r in b.itertuples()])
print('Documentos indexados:', col.count())

def mostrar(res):
    return pd.DataFrame({'texto': res['documents'][0], 'distancia_coseno': [d if ESPACIO == 'cosine' else d / 2 for d in res['distances'][0]],
                         'producto': [m['producto'] for m in res['metadatas'][0]],
                         'provincia': [m['provincia'] for m in res['metadatas'][0]],
                         'y_medio': [m['y_medio'] for m in res['metadatas'][0]]})

consulta = 'YPF estación de servicio GNC La Plata Buenos Aires'
q = np.asarray(embed([consulta]), dtype='float32')
inc = ['documents', 'distances', 'metadatas']

print('1) Consulta por texto:', consulta)
display(mostrar(col.query(query_embeddings=q.tolist(), n_results=5, include=inc)))

print('2) Misma consulta con filtro de metadatos (producto = GNC y provincia = BUENOS AIRES):')
display(mostrar(col.query(query_embeddings=q.tolist(), n_results=5, include=inc,
                          where={'$and': [{'producto': 'GNC'}, {'provincia': 'BUENOS AIRES'}]})))

print('3) Vecinos del documento de referencia:')
display(mostrar(col.query(query_embeddings=E[ref:ref + 1].tolist(), n_results=6, include=inc)))

# contraste con búsqueda exacta (fuerza bruta)
exacto = set(np.argsort(-(E @ q[0]))[:5]); aprox = set(int(i.split('_')[1]) for i in col.query(query_embeddings=q.tolist(), n_results=5)['ids'][0])
print(f'Coincidencia HNSW vs búsqueda exacta en el top-5: {len(exacto & aprox)}/5')

## 6. Redes neuronales artificiales: perceptrón y MLP (Keras)

### 6.0 Problema predictivo
**Predecir `log1p(volumen)` de un establecimiento–producto–canal en un período futuro**, usando únicamente información disponible **antes** de ese período. Es un problema de **regresión**, así que se usan MAE, MSE, RMSE y R² (las métricas de clasificación no aplican).

**Diseño anti-fuga de información (*data leakage*)**
- **Partición temporal** (no aleatoria): entrenamiento = períodos anteriores a 2023-01; validación = 2023-01; prueba = 2024-01 (el "futuro"). Una partición aleatoria mezclaría el futuro en el entrenamiento.
- **Predictores solo del pasado:** volumen y precio del período anterior observado de la misma clave, brecha en meses, nº de observaciones previas y media histórica. No se usan `volumen` ni (por defecto) el precio del mismo período.
- **Escalado, imputación y codificación se ajustan solo con el entrenamiento** (`ColumnTransformer.fit` sobre *train*) y se aplican a validación y prueba.
- Los clusters de la sección 4 **no** se usan como predictores (se calcularon con `y`).
- El conjunto de **prueba se usa una sola vez**, al final; la validación sirve para *early stopping* y elegir modelo.
- Se descartan `mes` y `anio`: solo hay enero (y un febrero), por lo que no hay estacionalidad estimable y `anio` obligaría a extrapolar.

In [ ]:
# Celda 19 · Rezagos (solo pasado), partición temporal y preprocesamiento sin fuga
g = panel.groupby(CLAVE, sort=False)
panel['lag_y'] = g['y'].shift(1)
panel['lag_log_precio'] = g['log_precio'].shift(1)
panel['lag_periodo'] = g['periodo'].shift(1)
panel['gap_meses'] = ((panel['periodo'].dt.year - panel['lag_periodo'].dt.year) * 12 +
                      (panel['periodo'].dt.month - panel['lag_periodo'].dt.month))
panel['n_previas'] = g.cumcount()
panel['media_hist_y'] = (g['y'].cumsum() - panel['y']) / panel['n_previas'].replace(0, np.nan)   # media de períodos ANTERIORES
panel['sin_historial'] = panel['lag_y'].isna().astype(int)

F_VAL, F_TEST = pd.Timestamp('2023-01-01'), pd.Timestamp('2024-01-01')
train = panel[panel['periodo'] < F_VAL].copy()
val = panel[panel['periodo'] == F_VAL].copy()
test = panel[panel['periodo'] == F_TEST].copy()
resumen = pd.DataFrame({'filas': [len(train), len(val), len(test)],
                        '% sin historial': [d['sin_historial'].mean() * 100 for d in (train, val, test)]},
                       index=['train (<2023-01)', 'val (2023-01)', 'test (2024-01)'])
display(resumen)

FEATS_NUM = ['lag_y', 'lag_log_precio', 'gap_meses', 'n_previas', 'media_hist_y']
if USAR_PRECIO_ACTUAL:
    FEATS_NUM.append('log_precio')
FEATS_CAT = ['producto', 'canal_de_comercializacion', 'provincia', 'bandera', 'tipo_negocio']
FEATS_BIN = ['exentos_i', 'sin_historial']

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), FEATS_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=30, sparse_output=False), FEATS_CAT),
    ('bin', 'passthrough', FEATS_BIN)])

cols = FEATS_NUM + FEATS_CAT + FEATS_BIN
Xtr = pre.fit_transform(train[cols]).astype('float32')       # fit SOLO con entrenamiento
Xva = pre.transform(val[cols]).astype('float32')
Xte = pre.transform(test[cols]).astype('float32')
ytr, yva, yte = train['y'].values, val['y'].values, test['y'].values
SPLITS = {'train': (train, Xtr, ytr), 'val': (val, Xva, yva), 'test': (test, Xte, yte)}
nombres_X = list(pre.get_feature_names_out())
print('Dimensión de entrada:', Xtr.shape[1], '| nulos tras el preprocesamiento:', int(np.isnan(Xtr).sum() + np.isnan(Xva).sum() + np.isnan(Xte).sum()))

In [ ]:
# Celda 20 · Líneas base, métricas y modelo de referencia (árboles)
def metricas(y_log, p_log):
    p_log = np.clip(p_log, 0, None)                  # el volumen no puede ser negativo
    yv, pv = np.expm1(y_log), np.expm1(p_log)
    return {'MAE_log': mean_absolute_error(y_log, p_log),
            'MSE_log': mean_squared_error(y_log, p_log),
            'RMSE_log': np.sqrt(mean_squared_error(y_log, p_log)),
            'R2_log': r2_score(y_log, p_log),
            'MAE': mean_absolute_error(yv, pv),
            'RMSE': np.sqrt(mean_squared_error(yv, pv)),
            'R2': r2_score(yv, pv)}

med_prod = train.groupby('producto')['y'].median()
med_glob = train['y'].median()
base_mediana = lambda d: d['producto'].map(med_prod).fillna(med_glob).values
base_persist = lambda d: np.where(d['lag_y'].notna(), d['lag_y'], base_mediana(d))   # "repite el último valor observado"

resultados = {}
resultados['Baseline 1: mediana por producto'] = {s: base_mediana(d) for s, (d, X, y) in SPLITS.items()}
resultados['Baseline 2: persistencia (último valor)'] = {s: base_persist(d) for s, (d, X, y) in SPLITS.items()}

hgb = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=SEMILLA).fit(Xtr, ytr)
resultados['Referencia: Gradient Boosting'] = {s: hgb.predict(X) for s, (d, X, y) in SPLITS.items()}
print('Modelos registrados:', list(resultados))

### 6.1 Perceptrón simple
**Arquitectura:** una sola neurona sin capas ocultas. Dado el vector de entrada $x\in\mathbb{R}^p$:
$$\hat y=\sigma(w^\top x+b),\qquad \sigma(z)=\frac{1}{1+e^{-z}}$$
- **Pesos $w$** ponderan la importancia de cada variable; **sesgo $b$** desplaza la frontera de decisión.
- **Sigmoide:** comprime la salida a $(0,1)$. Por eso el objetivo se reescala con *min-max* usando solo el entrenamiento y la predicción se vuelve a la escala original. **Limitación:** un volumen de prueba mayor que el máximo de entrenamiento no puede alcanzarse (la salida satura en 1).
- **Entrenamiento:** se minimiza el error cuadrático medio por descenso de gradiente (optimizador Adam); en cada paso $w\leftarrow w-\eta\,\partial L/\partial w$.
- **Alcance:** al ser un modelo lineal en $w^\top x$ con una no linealidad monótona, solo captura relaciones aproximadamente lineales (en el espacio de la sigmoide).

In [ ]:
# Celda 21 · Perceptrón simple (Keras)
import tensorflow as tf
from tensorflow import keras
layers, regularizers, callbacks = keras.layers, keras.regularizers, keras.callbacks
keras.utils.set_random_seed(SEMILLA)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)

y_min, y_max = ytr.min(), ytr.max()                          # escalado min-max con datos de entrenamiento
esc = lambda y: (y - y_min) / (y_max - y_min)
inv = lambda p: p * (y_max - y_min) + y_min

perceptron = keras.Sequential([keras.Input(shape=(Xtr.shape[1],)),
                               layers.Dense(1, activation='sigmoid', name='neurona')])
perceptron.compile(optimizer=keras.optimizers.Adam(1e-2), loss='mse', metrics=['mae'])
es = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
h_perc = perceptron.fit(Xtr, esc(ytr), validation_data=(Xva, esc(yva)),
                        epochs=200, batch_size=256, callbacks=[es], verbose=0)
print('Épocas entrenadas:', len(h_perc.history['loss']))

resultados['Perceptrón simple (sigmoide)'] = {s: inv(perceptron.predict(X, verbose=0).ravel()) for s, (d, X, y) in SPLITS.items()}

w, b = perceptron.layers[0].get_weights()
pesos = pd.Series(w.ravel(), index=nombres_X).sort_values(key=np.abs, ascending=False)
print('Sesgo b =', float(b[0]))
print('Variables con mayor peso |w|:'); display(pesos.head(8).to_frame('peso'))

### 6.2 Perceptrón multicapa (MLP)
**Arquitectura:** capa de entrada ($p$ variables) → capas ocultas densas con **ReLU** ($128\to64\to32$) con *Dropout* y regularización L2 → capa de salida lineal de 1 neurona (regresión). Función de pérdida: **MSE**; optimizador: **Adam** (descenso por gradiente con tasas adaptativas por parámetro).

**Cómo aprende**
1. **Forward propagation:** para cada capa $l$, $z^{(l)}=W^{(l)}a^{(l-1)}+b^{(l)}$ y $a^{(l)}=f(z^{(l)})$. La salida $\hat y$ se compara con $y$ mediante la pérdida $L=\tfrac12(\hat y-y)^2$.
2. **Backpropagation:** regla de la cadena de la salida hacia la entrada. En la capa de salida $\delta^{(L)}=\partial L/\partial z^{(L)}$; en las ocultas $\delta^{(l)}=\big(W^{(l+1)\top}\delta^{(l+1)}\big)\odot f'(z^{(l)})$. Los gradientes son $\partial L/\partial W^{(l)}=\delta^{(l)}a^{(l-1)\top}$ y $\partial L/\partial b^{(l)}=\delta^{(l)}$.
3. **Actualización:** $W\leftarrow W-\eta\,\partial L/\partial W$ (Adam además usa momentos de primer y segundo orden del gradiente). Se repite por lotes (*mini-batches*) durante varias épocas.

La celda siguiente implementa **a mano con NumPy** una red pequeña, verifica los gradientes de backpropagation con **diferencias finitas** y ejecuta un paso de descenso: es el mismo mecanismo que Keras automatiza.

In [ ]:
# Celda 22 · Forward + backpropagation a mano (NumPy) con verificación numérica del gradiente
rng = np.random.default_rng(0)
x = np.array([[0.5, -1.0]]); t = np.array([[0.7]])
W1 = rng.normal(size=(2, 3)) * 0.5; b1 = np.zeros((1, 3))
W2 = rng.normal(size=(3, 1)) * 0.5; b2 = np.zeros((1, 1))
sig = lambda z: 1 / (1 + np.exp(-z))

def forward(W1, b1, W2, b2):
    z1 = x @ W1 + b1; a1 = sig(z1)                    # capa oculta (sigmoide)
    z2 = a1 @ W2 + b2; yhat = z2                      # salida lineal
    return z1, a1, yhat, 0.5 * np.sum((yhat - t) ** 2)

z1, a1, yhat, L = forward(W1, b1, W2, b2)
dz2 = yhat - t                                        # dL/dz2
dW2, db2 = a1.T @ dz2, dz2
da1 = dz2 @ W2.T
dz1 = da1 * a1 * (1 - a1)                             # f'(z)=a(1-a) para la sigmoide
dW1, db1 = x.T @ dz1, dz1

eps = 1e-6                                            # gradiente numérico de W1[0,0]
Wp, Wm = W1.copy(), W1.copy(); Wp[0, 0] += eps; Wm[0, 0] -= eps
g_num = (forward(Wp, b1, W2, b2)[3] - forward(Wm, b1, W2, b2)[3]) / (2 * eps)
print(f'Pérdida inicial: {L:.6f}')
print(f'dL/dW1[0,0]  backprop = {dW1[0,0]:.8f} | numérico = {g_num:.8f} | diferencia = {abs(dW1[0,0]-g_num):.2e}')

lr = 0.5
W1n, b1n, W2n, b2n = W1 - lr * dW1, b1 - lr * db1, W2 - lr * dW2, b2 - lr * db2
print(f'Pérdida tras un paso de descenso (lr={lr}): {forward(W1n, b1n, W2n, b2n)[3]:.6f}  (menor que la inicial)')

In [ ]:
# Celda 23 · MLP (Keras)
mu, sd = ytr.mean(), ytr.std()                               # estandarización del objetivo con datos de entrenamiento

def construir_mlp(n_in, unidades=(128, 64, 32), drop=0.2, l2=1e-4, lr=1e-3):
    m = keras.Sequential([keras.Input(shape=(n_in,))])
    for u in unidades:
        m.add(layers.Dense(u, activation='relu', kernel_regularizer=regularizers.l2(l2)))
        m.add(layers.Dropout(drop))
    m.add(layers.Dense(1))                                   # salida lineal (regresión)
    m.compile(optimizer=keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return m

keras.utils.set_random_seed(SEMILLA)
mlp = construir_mlp(Xtr.shape[1])
mlp.summary()
cbs = [callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
       callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-5)]
h_mlp = mlp.fit(Xtr, (ytr - mu) / sd, validation_data=(Xva, (yva - mu) / sd),
                epochs=300, batch_size=256, callbacks=cbs, verbose=0)
print('Épocas entrenadas:', len(h_mlp.history['loss']), '| mejor val_loss:', round(min(h_mlp.history['val_loss']), 4))

resultados['MLP (128-64-32, dropout + L2)'] = {s: mlp.predict(X, verbose=0).ravel() * sd + mu for s, (d, X, y) in SPLITS.items()}

# Modelo deliberadamente sobredimensionado y sin regularización, para ilustrar el sobreajuste
keras.utils.set_random_seed(SEMILLA)
mlp_grande = construir_mlp(Xtr.shape[1], unidades=(256, 256, 256), drop=0.0, l2=0.0)
h_big = mlp_grande.fit(Xtr, (ytr - mu) / sd, validation_data=(Xva, (yva - mu) / sd),
                       epochs=80, batch_size=256, verbose=0)
resultados['MLP grande sin regularización (sobreajuste)'] = {s: mlp_grande.predict(X, verbose=0).ravel() * sd + mu for s, (d, X, y) in SPLITS.items()}

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a, h, tit in zip(ax, [h_perc, h_mlp, h_big], ['Perceptrón (escala 0–1)', 'MLP regularizado', 'MLP grande sin regularización']):
    a.plot(h.history['loss'], label='entrenamiento'); a.plot(h.history['val_loss'], label='validación')
    a.set_title(tit); a.set_xlabel('época'); a.set_ylabel('MSE'); a.legend()
plt.tight_layout(); plt.show()

### 6.3 Evaluación
Se evalúa en las tres particiones con MAE, MSE, RMSE y R². Las métricas con sufijo `_log` están en la escala del modelo (`log1p(volumen)`): un MAE_log de 0,5 equivale, aproximadamente, a equivocarse por un factor multiplicativo de $e^{0{,}5}\approx1{,}65$. Las métricas **sin sufijo** están en la escala original del volumen y **las dominan unos pocos registros enormes (GNC, gasoil a granel)**; por eso el R² original puede ser bajo o negativo aunque el modelo sea razonable en la mayoría de los casos.

**Ojo con la composición de las particiones:** en entrenamiento ≈ 46 % de las filas no tiene historial (son los primeros registros de cada clave), mientras que en validación y prueba solo ≈ 6 % y ≈ 4 %. Por eso se reporta también el error separando filas **con** y **sin** historial.

In [ ]:
# Celda 24 · Tabla de métricas por modelo y partición
filas = []
for nombre, preds in resultados.items():
    for sp, (d, X, y) in SPLITS.items():
        filas.append({'modelo': nombre, 'split': sp, **metricas(y, preds[sp])})
tabla = pd.DataFrame(filas)

for sp in ['val', 'test']:
    print(f'\n=== {sp.upper()} ===')
    display(tabla[tabla.split == sp].set_index('modelo').drop(columns='split').sort_values('RMSE_log'))

print('\n=== TEST solo filas CON historial (lag disponible) ===')
m_h = (test['sin_historial'] == 0).values
display(pd.DataFrame({n: metricas(yte[m_h], p['test'][m_h]) for n, p in resultados.items()}).T[['MAE_log', 'RMSE_log', 'R2_log']].sort_values('RMSE_log'))
print('=== TEST solo filas SIN historial (primer registro de la clave) ===')
display(pd.DataFrame({n: metricas(yte[~m_h], p['test'][~m_h]) for n, p in resultados.items()}).T[['MAE_log', 'RMSE_log', 'R2_log']].sort_values('RMSE_log'))

In [ ]:
# Celda 25 · Sobreajuste / subajuste y capacidad de generalización
piv = tabla.pivot(index='modelo', columns='split', values='RMSE_log')[['train', 'val', 'test']]
piv['val - train'] = piv['val'] - piv['train']
piv['test - val'] = piv['test'] - piv['val']
display(piv.sort_values('test'))

piv.drop(columns=['val - train', 'test - val']).plot(kind='bar', figsize=(11, 4.5))
plt.ylabel('RMSE (escala log)'); plt.title('Error por partición: la brecha train→val/test indica sobreajuste o cambio de distribución')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Celda 26 · Diagnóstico de residuos de la mejor red neuronal según validación
elegibles = [m for m in resultados if m.startswith(('Perceptrón', 'MLP (')) ]
if not elegibles:                                    # si aún no se ejecutaron las celdas de Keras
    elegibles = [m for m in resultados if not m.startswith('Baseline')]
mejor = tabla[(tabla.split == 'val') & tabla.modelo.isin(elegibles)].set_index('modelo')['RMSE_log'].idxmin()
print('Mejor modelo neuronal (por RMSE_log en validación):', mejor)
p = np.clip(resultados[mejor]['test'], 0, None); res = yte - p

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].scatter(yte, p, s=4, alpha=.3); lim = [0, max(yte.max(), p.max())]; ax[0].plot(lim, lim, 'r--')
ax[0].set_xlabel('real (log1p)'); ax[0].set_ylabel('predicho'); ax[0].set_title('Real vs predicho (test)')
ax[1].hist(res, bins=60, color='#55a868'); ax[1].axvline(0, color='r'); ax[1].set_title(f'Residuos (sesgo medio = {res.mean():.3f})')
err_prod = pd.Series(np.abs(res), index=test['producto'].values).groupby(level=0).mean().sort_values()
err_prod.plot(kind='barh', ax=ax[2]); ax[2].set_title('MAE_log por producto (test)')
plt.tight_layout(); plt.show()

# aciertos aproximados: % de predicciones dentro de un factor 1,5 y 2 del valor real
razon = np.expm1(p) / np.expm1(yte)
print(f'Dentro de x1,5 del valor real: {np.mean((razon > 1/1.5) & (razon < 1.5)):.1%} | dentro de x2: {np.mean((razon > 0.5) & (razon < 2)):.1%}')

## 7. Soporte teórico de los errores

**Métricas de regresión** (con $n$ observaciones, $e_i=y_i-\hat y_i$):

| Métrica | Fórmula | Lectura |
|---|---|---|
| MAE | $\frac1n\sum\lvert e_i\rvert$ | Error medio en las unidades del objetivo; robusto a atípicos. |
| MSE | $\frac1n\sum e_i^2$ | Penaliza cuadráticamente los errores grandes; es la pérdida que se optimiza. |
| RMSE | $\sqrt{\text{MSE}}$ | Misma unidad que el objetivo; más sensible a atípicos que el MAE. RMSE ≫ MAE indica errores grandes aislados. |
| R² | $1-\frac{\sum e_i^2}{\sum (y_i-\bar y)^2}$ | Proporción de varianza explicada **respecto de predecir siempre la media**. Puede ser negativo en datos nuevos. |

**Error en escala logarítmica.** Un error $e$ en $\log(1+y)$ equivale a un error **relativo** de $\approx e^{e}-1$. Por eso el RMSE en escala log trata igual a un error del 50 % en una estación chica o grande, mientras que el RMSE original lo domina el volumen absoluto. La transformación inversa $\exp(\hat y)-1$ de un promedio en log estima la **mediana** condicional, no la media (se subestima la media si la distribución es asimétrica).

**Descomposición del error:** $\mathbb{E}[(y-\hat f)^2]=\text{sesgo}^2+\text{varianza}+\sigma^2_{\text{irreducible}}$.
- **Subajuste (alto sesgo):** error alto en train **y** en validación; el modelo es demasiado simple (p. ej. el perceptrón frente al MLP). Acción: más capacidad o mejores variables.
- **Sobreajuste (alta varianza):** error muy bajo en train y notablemente mayor en validación; la curva de validación sube mientras la de entrenamiento baja. Acciones: *early stopping*, Dropout, L2, menos parámetros, más datos.
- **Cambio de distribución (*drift*):** si el error de **prueba** supera al de validación sin señales de sobreajuste, el futuro difiere del pasado (precios nominales que cambian por inflación, otra composición de estaciones). Es el riesgo central de pronosticar con datos de pocos períodos.

**Fuga de información (*data leakage*).** Es un error de *diseño*, no de cálculo: métricas anormalmente buenas en validación y malas en producción. Fuentes típicas: ajustar el escalador/imputador con todos los datos, partir al azar series temporales, usar variables del mismo período que no se conocen al predecir, o usar clusters calculados con el objetivo.

**Línea base.** Un modelo solo aporta valor si supera a la **persistencia** ("el volumen será el del último período observado"), que es muy competitiva en series con memoria. Mejorar sobre la mediana por producto es una condición necesaria pero débil.

**Incertidumbre.** Con 8 períodos y cobertura desigual, las diferencias pequeñas entre modelos pueden no ser significativas; conviene repetir con varias semillas y, si hubiera más períodos, usar validación cruzada temporal (*rolling origin*).

| Síntoma | Diagnóstico probable | Qué hacer |
|---|---|---|
| Train ≪ val ≈ test | Sobreajuste | Regularizar, simplificar, parar antes |
| Train ≈ val ≈ test, todos altos | Subajuste o falta de señal | Más variables, más capacidad |
| val bajo, test alto | Drift temporal | Revisar variables nominales (precios), reentrenar con datos recientes |
| Resultado "demasiado bueno" | Posible fuga | Auditar variables y orden del *fit* del preprocesamiento |
| R² original negativo, R² log positivo | Unos pocos valores gigantes dominan | Reportar ambas escalas; revisar atípicos y unidades |

In [ ]:
# Celda 27 · Resumen automático de resultados
t_test = tabla[tabla.split == 'test'].set_index('modelo')
pers = t_test.loc['Baseline 2: persistencia (último valor)', 'RMSE_log']
print('RMSE_log en TEST (menor es mejor):')
for n, r in t_test['RMSE_log'].sort_values().items():
    print(f'  {n:<48s} {r:6.3f}   ({(1 - r / pers) * 100:+6.1f}% vs persistencia)')
print(f'\nModelo elegido por validación: {mejor}')
print(f"RMSE_log test = {t_test.loc[mejor,'RMSE_log']:.3f} | MAE_log = {t_test.loc[mejor,'MAE_log']:.3f} | R2_log = {t_test.loc[mejor,'R2_log']:.3f}")

## 8. Conclusiones, limitaciones y próximos pasos

**Qué se puede afirmar con rigor**
- El conjunto tiene **solo 8 períodos** y cobertura muy desigual por establecimiento: el "futuro" evaluado es **2024-01** a partir de datos previos, no un pronóstico mensual continuo.
- La estructura de los datos está dominada por **producto** (GNC con magnitudes de volumen mucho mayores), por el **tipo de negocio** y por registros de **precio atípicamente alto**; SVD resume el 91 % de la varianza en 7 componentes.
- **K-means** (k = 5) segmenta mejor el mercado; **DBSCAN** fragmenta en decenas de grupos puros y es más útil para **detectar observaciones extremas**.
- Comparar los modelos neuronales **siempre contra la persistencia**: es la referencia que importa para estimar volumen futuro.

**Limitaciones**
- La **unidad de `volumen` y de los precios altos no está documentada**; no se asumió ninguna.
- Sin estacionalidad estimable (casi todo es enero) y con inflación en los precios nominales (riesgo de *drift*).
- Las filas **sin historial** (primer registro de una clave) solo pueden predecirse con atributos estáticos; su error es mayor.
- Muestra posiblemente no representativa de todo el mercado.

**Próximos pasos**
1. Documentar/verificar unidades de volumen y precio con la fuente; revisar los atípicos que marcó DBSCAN.
2. Incorporar más períodos mensuales para estacionalidad y validación cruzada temporal.
3. Probar pérdida Huber, embeddings de categorías en la red (en lugar de one-hot) y vecinos de ChromaDB como variable de contexto (cuidando de calcularlos solo con datos de entrenamiento).
4. Intervalos de predicción (regresión cuantílica o *dropout* en inferencia).

**Problemas frecuentes al ejecutar en Colab**

| Mensaje / síntoma | Causa y solución |
|---|---|
| `ModuleNotFoundError: chromadb` o `sentence_transformers` | Ejecutar la celda 1 y, si lo pide, `Entorno de ejecución → Reiniciar sesión`. |
| `FileNotFoundError: EESS_limpio.csv` | Subir el archivo cuando la celda 3 lo pida o montarlo desde Drive y cambiar `RUTA`. |
| `TypeError` por `sparse_output` en `OneHotEncoder` | scikit-learn < 1.2: cambiar `sparse_output=False` por `sparse=False`. |
| Descarga del modelo de embeddings falla (sin internet) | Poner `USAR_SENTENCE_TRANSFORMERS = False` (usa TF-IDF + SVD). |
| Kernel sin memoria en DBSCAN | Reducir `N_MUESTRA` o aumentar `min_samples`/disminuir `eps`. |